# CRIVO: Transformer próprio como leitor das fichas

O Transformer do CRIVO (pesos treinados do zero, nenhum modelo externo) aprende a dizer **se um fato da ficha responde a uma pergunta**, mesmo com outras palavras ("o quadro mais famoso" ↔ "pintou Abaporu").

O ponto de partida é o **pré-treino em português de 16M** que o caderno `treinar_transformer_16m_colab.ipynb` gravou no seu Drive. Não é preciso pré-treinar de novo.

**Antes de começar:** Ambiente de execução → Alterar tipo → **GPU**. Execute as células na ordem.

O leitor só fica ligado se passar em duas provas:
1. melhorar a validação com as perguntas do tutor, que ele nunca vê no treino;
2. melhorar o teste congelado v2 (`avaliacoes/leitura_ficha_v2`), escrito antes deste treino.

Se não passar, nada muda no CRIVO.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, shutil, subprocess, sys

drive.mount('/content/drive')
BRANCH = 'main'   # troque se o código estiver em outra branch
ROOT = Path('/content/CRIVO')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/HROSONE/CRIVO.git', str(ROOT)], check=True)
SAIDA = Path('/content/drive/MyDrive/CRIVO/leitor-transformer')
SAIDA.mkdir(parents=True, exist_ok=True)
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=ROOT, capture_output=True, text=True).stdout)

## Achar o pré-treino de 16M no Drive

Procura `pretreino/melhor/pesos.pt` dentro de `MyDrive/CRIVO/transformer-16m`. Se você guardou em outro lugar, escreva o caminho da pasta em `BASE`.

In [ ]:
BASE = None   # ex.: '/content/drive/MyDrive/CRIVO/transformer-16m/<experimento>/pretreino/melhor'
if BASE is None:
    achados = sorted(Path('/content/drive/MyDrive/CRIVO/transformer-16m').glob('**/pretreino/melhor/pesos.pt'),
                     key=lambda p: p.stat().st_mtime)
    if not achados:
        raise RuntimeError('Não achei pretreino/melhor/pesos.pt no Drive. Rode antes o pré-treino do caderno treinar_transformer_16m_colab.ipynb ou defina BASE.')
    BASE = str(achados[-1].parent)
print('Pré-treino:', BASE)
print(json.dumps(json.loads((Path(BASE) / 'relatorio.json').read_text()).get('config', {}), ensure_ascii=False))

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.24,<3'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ative a GPU: Ambiente de execução → Alterar tipo → GPU.')
print(torch.__version__, torch.cuda.get_device_name(0))

## Linha de base (sem o Transformer)

Mede a leitura atual no teste congelado v2, antes de qualquer mudança.

In [ ]:
base = subprocess.run([sys.executable, 'scripts/avaliar_leitura_ficha.py', 'teste_v2'], cwd=ROOT, capture_output=True, text=True, check=True)
linha_base = json.loads(base.stdout.strip().splitlines()[-1])
print(linha_base)

## Treinar o leitor

Cerca de 30 a 60 minutos numa T4. O melhor passo, medido nas perguntas do tutor, é gravado no Drive. Se a sessão cair, rode de novo a partir da primeira célula.

In [ ]:
cmd = [sys.executable, '-u', 'scripts/treinar_leitor_transformer.py', '--base', BASE,
       '--dispositivo', 'cuda', '--passos', '4000', '--lote', '64', '--lr', '5e-5',
       '--avaliar-a-cada', '250', '--saida', str(SAIDA / 'leitor_transformer')]
subprocess.run(cmd, cwd=ROOT, check=True)
print(json.loads((SAIDA / 'leitor_transformer/meta.json').read_text())['validacao_tutor'])

## Prova 1: a leitura melhora com o Transformer?

Validação cruzada com as perguntas do tutor. O leitor só é aprovado se entregar mais fatos certos sem mais erros.

In [ ]:
subprocess.run([sys.executable, 'scripts/treinar_leitura_ficha.py', '--com-transformer', str(SAIDA / 'leitor_transformer'),
                '--saida', str(SAIDA / 'leitura_ficha')], cwd=ROOT, check=True)
meta_lt = json.loads((SAIDA / 'leitor_transformer/meta.json').read_text())
print('Aprovado na validação:', meta_lt['controle']['aprovado'])

## Prova 2: teste congelado v2

Copia os artefatos para o código e mede de novo. Só fica se o teste congelado melhorar.

In [ ]:
if not meta_lt['controle']['aprovado']:
    print('Não passou na prova 1: o CRIVO continua como está. Compartilhe mesmo assim o arquivo resultado.json abaixo.')
    resultado = {'linha_base_v2': linha_base, 'aprovado_validacao': False, 'validacao': meta_lt['controle']}
else:
    shutil.copytree(SAIDA / 'leitor_transformer', ROOT / 'artefatos/leitor_transformer', dirs_exist_ok=True)
    shutil.copyfile(SAIDA / 'leitura_ficha/meta.json', ROOT / 'artefatos/leitura_ficha/meta.json')
    com = subprocess.run([sys.executable, 'scripts/avaliar_leitura_ficha.py', 'teste_v2'], cwd=ROOT, capture_output=True, text=True, check=True)
    com_transformer = json.loads(com.stdout.strip().splitlines()[-1])
    def certos(r): return r['respondiveis'].get('afirmou_certo', 0) + r['respondiveis'].get('aproximou_certo', 0)
    def erros(r): return r['respondiveis'].get('afirmou_errado', 0) + r['sem_resposta'].get('afirmou', 0)
    fica = certos(com_transformer) > certos(linha_base) and erros(com_transformer) <= erros(linha_base)
    resultado = {'linha_base_v2': linha_base, 'com_transformer_v2': com_transformer,
                 'aprovado_validacao': True, 'fica_ligado': fica, 'validacao': meta_lt['controle']}
    print('Sem Transformer:', certos(linha_base), 'certos,', erros(linha_base), 'erros')
    print('Com Transformer:', certos(com_transformer), 'certos,', erros(com_transformer), 'erros')
    print('FICA LIGADO' if fica else 'NÃO MELHOROU NO TESTE CONGELADO: o CRIVO continua como está')
(SAIDA / 'resultado.json').write_text(json.dumps(resultado, ensure_ascii=False, indent=1))

## Levar o resultado para o CRIVO

A célula abaixo cria `leitor_transformer.zip` no Drive (`MyDrive/CRIVO/leitor-transformer/`). Ele traz os pesos, a leitura recalibrada e o `resultado.json`.

Envie esse zip na conversa com o Claude. Ele confere os números, roda os testes e abre o PR. Se o resultado for "não melhorou", envie só o `resultado.json`.

In [ ]:
pacote = Path('/content/pacote'); shutil.rmtree(pacote, ignore_errors=True); pacote.mkdir()
shutil.copytree(SAIDA / 'leitor_transformer', pacote / 'leitor_transformer')
if (SAIDA / 'leitura_ficha/meta.json').exists():
    (pacote / 'leitura_ficha').mkdir(); shutil.copyfile(SAIDA / 'leitura_ficha/meta.json', pacote / 'leitura_ficha/meta.json')
shutil.copyfile(SAIDA / 'resultado.json', pacote / 'resultado.json')
shutil.make_archive(str(SAIDA / 'leitor_transformer'), 'zip', pacote)
print('Pronto:', SAIDA / 'leitor_transformer.zip')